17.11.2026

## 6.1

Дан массив arr = np.arange(20) получите :

• каждый третий элемент, начиная с индекса 1

• элементы с 5-го по 15-й (не включая 15) в обратном порядке

• последние 5 элементов, идущих в обратном порядке

In [1]:
import numpy as np

arr = np.arange(20)

print(arr[1::3])
print(arr[5:15][::-1])
print(arr[-5:][::-1])

[ 1  4  7 10 13 16 19]
[14 13 12 11 10  9  8  7  6  5]
[19 18 17 16 15]


## 6.2

Создайте матрицу 6×6 случайных целых чисел от 0 до 9. Извлеките:

• подматрицу из строк 2–4 и столбцов 1–3 (включительно)

• главную диагональ через np.diagonal

• «шахматный» узор: элементы с чётной суммой индексов

In [2]:
import numpy as np

np.random.seed(1520441)
A = np.random.randint(0, 10, (6, 6))

print("Матрица:")
print(A)
print("Подматрица:")
print(A[2:5, 1:4])
print("Главная диагональ:", np.diagonal(A))

rows, columns = np.indices(A.shape)
mask = (rows + columns) % 2 == 0
print("Элементы шахматного узора:", A[mask])

Матрица:
[[2 1 3 6 3 7]
 [6 3 4 8 3 7]
 [7 7 2 7 8 3]
 [3 1 6 9 6 5]
 [8 7 6 1 1 2]
 [7 8 0 5 6 1]]
Подматрица:
[[7 2 7]
 [1 6 9]
 [7 6 1]]
Главная диагональ: [2 3 2 9 1 1]
Элементы шахматного узора: [2 3 3 3 8 7 7 2 8 1 9 5 8 6 1 8 5 1]


## 6.3

Дан массив a = np.array([10, 20, 30, 40, 50, 60]) и матрица m = np.arange(12).reshape(3, 4).
Решите:

• выбрать элементы a по индексам [0, 2, 4];

• из матрицы m выбрать элементы (0,1), (1,3), (2,0) - то есть «попарно» индексы строк и столбцов

• из матрицы m выбрать блок из строк [0, 2] и столбцов [1, 3] в виде матрицы 2×2 (а не 4 элементов)

In [3]:
import numpy as np

a = np.array([10, 20, 30, 40, 50, 60])
m = np.arange(12).reshape(3, 4)

print("Элементы a:", a[[0, 2, 4]])
print("Попарный выбор:", m[[0, 1, 2], [1, 3, 0]])
print("Блок 2×2:")
print(m[np.ix_([0, 2], [1, 3])])

Элементы a: [10 30 50]
Попарный выбор: [1 7 8]
Блок 2×2:
[[ 1  3]
 [ 9 11]]


## 6.4

Дан массив x = np.random.randn(1000). Решите:

• подсчитайте количество элементов, больших 1

• замените все отрицательные элементы на 0

• обнулите элементы, стоящие на позициях, где соседи (слева и справа) имеют разные знаки (т.е. где происходит смена знака)

In [4]:
import numpy as np

np.random.seed(1520441)
x = np.random.randn(1000)

print("Количество элементов больше 1:", np.sum(x > 1))

y = x.copy()
y[y < 0] = 0
print("Количество отрицательных элементов:", np.sum(y < 0))

z = x.copy()
positions = np.where(x[:-2] * x[2:] < 0)[0] + 1
z[positions] = 0
print("Обнулено позиций:", len(positions))

Количество элементов больше 1: 154
Количество отрицательных элементов: 0
Обнулено позиций: 506


## 7.1

Создайте на диске файл big.dat с массивом формы (50000, 200) типа float32. Заполните его случайными числами. Убедитесь, что файл содержит ровно50000 × 200 × 4 байт

In [5]:
import numpy as np
from pathlib import Path

file = Path("big.dat")
mode = "r+" if file.exists() else "w+"
m = np.memmap(file, dtype="float32", mode=mode, shape=(50000, 200))
rng = np.random.default_rng(1520441)

for start in range(0, 50000, 1000):
    m[start:start + 1000] = rng.random((1000, 200), dtype=np.float32)

m.flush()
size = file.stat().st_size
print("Размер файла:", size, "байт")
print("Ожидаемый размер:", 50000 * 200 * 4, "байт")
print("Размер совпадает:", size == 50000 * 200 * 4)

Размер файла: 40000000 байт
Ожидаемый размер: 40000000 байт
Размер совпадает: True


## 7.2

Откройте big.dat в режиме только для чтения. Вычислите построчно (батчами) глобальные min, max и среднее значение по всему массиву. Не используйте variable[:] - это загрузиn весь массив

In [6]:
import numpy as np

m = np.memmap("big.dat", dtype="float32", mode="r", shape=(50000, 200))
minimum = float("inf")
maximum = float("-inf")
total = 0.0
count = 0

for start in range(0, 50000, 1000):
    block = m[start:start + 1000]
    minimum = min(minimum, block.min())
    maximum = max(maximum, block.max())
    total += block.sum(dtype=np.float64)
    count += block.size

print("Минимум:", minimum)
print("Максимум:", maximum)
print("Среднее:", total / count)

Минимум: 2.9802322e-07
Максимум: 0.99999994
Среднее: 0.5000275543777645


## 7.3

Создайте массив 3000×3000 (С-order). Вычислите сумму  построчно и постолбцово, используя чистые Python-циклы с доступом через arr[i, j]. Замерьте время. Затем проделайте то же для массива, созданного в F-order

In [7]:
import numpy as np
from time import perf_counter

for order in ["C", "F"]:
    arr = np.ones((3000, 3000), order=order)
    print("Порядок:", order)

    start = perf_counter()
    s = 0.0
    for i in range(3000):
        for j in range(3000):
            s += arr[i, j]
    elapsed = perf_counter() - start

    print("По строкам:", elapsed, "с")
    print("Сумма:", s)

    start = perf_counter()
    s = 0.0
    for j in range(3000):
        for i in range(3000):
            s += arr[i, j]
    elapsed = perf_counter() - start

    print("По столбцам:", elapsed, "с")
    print("Сумма:", s)

Порядок: C
По строкам: 1.1501025999896228 с
Сумма: 9000000.0
По столбцам: 1.1334631999779958 с
Сумма: 9000000.0
Порядок: F
По строкам: 1.1583579999860376 с
Сумма: 9000000.0
По столбцам: 1.1355190000031143 с
Сумма: 9000000.0


## 7.4

На массиве 10 млн элементов float64 сравните время вычисления (a + b) / (a - b + 1) тремя способами:

• Python-цикл с list

• NumPy-выражение в одну строку

• NumPy-выражение с out= для переиспользования буфера

In [8]:
import numpy as np
from time import perf_counter

a = np.ones(10**7, dtype="float64")
b = np.full(10**7, 0.5, dtype="float64")

# 1. Python-цикл со списками
a_list = a.tolist()
b_list = b.tolist()

start = perf_counter()

result_list = []
for i in range(len(a_list)):
    result_list.append(
        (a_list[i] + b_list[i]) / (a_list[i] - b_list[i] + 1)
    )

print("Python:", perf_counter() - start, "с")

# 2. NumPy в одну строку
start = perf_counter()

result_numpy = (a + b) / (a - b + 1)

print("NumPy:", perf_counter() - start, "с")

# 3. NumPy с повторным использованием буферов
result_out = np.empty_like(a)
denominator = np.empty_like(a)

start = perf_counter()

np.add(a, b, out=result_out)
np.subtract(a, b, out=denominator)
np.add(denominator, 1, out=denominator)
np.divide(result_out, denominator, out=result_out)

print("NumPy с out:", perf_counter() - start, "с")

Python: 1.5008854999905452 с
NumPy: 0.08859080000547692 с
NumPy с out: 0.03664279999793507 с


## 8.1

Даны списки row, col, data длиной 10 000. Соберите матрицу 5000×5000 в формате COO, преобразуйте в CSR, проверьте, что nnz (число ненулевых) корректен с учётом возможных совпадающих индексов

In [9]:
import numpy as np
from scipy import sparse

rng = np.random.default_rng(1520441)
row = rng.integers(0, 5000, 10000).tolist()
col = rng.integers(0, 5000, 10000).tolist()
data = rng.integers(1, 10, 10000).tolist()

row[1] = row[0]
col[1] = col[0]

S_coo = sparse.coo_matrix((data, (row, col)), shape=(5000, 5000))
S_csr = S_coo.tocsr()
unique_positions = len(set(zip(row, col)))

print("Записей в COO:", S_coo.nnz)
print("Ненулевых элементов в CSR:", S_csr.nnz)
print("Уникальных координат:", unique_positions)
print("Количество совпадает:", S_csr.nnz == unique_positions)

Записей в COO: 10000
Ненулевых элементов в CSR: 9998
Уникальных координат: 9998
Количество совпадает: True


## 8.2

Создайте матрицу 4000×4000 с плотностью 0.5%. Сравните:

• объём памяти для dense (float64) и CSR

• время умножения на вектор для обеих форм

In [10]:
import numpy as np
from scipy import sparse

S = sparse.random(4000, 4000, density=0.005, format="csr", dtype=np.float64, random_state=1520441)
A = S.toarray()
x = np.ones(4000)

memory_csr = S.data.nbytes + S.indices.nbytes + S.indptr.nbytes
print("Память dense:", A.nbytes, "байт")
print("Память CSR:", memory_csr, "байт")
print("Результаты совпадают:", np.allclose(S @ x, A @ x))

print("CSR:")
%timeit -n 20 -r 3 S @ x
print("Dense:")
%timeit -n 20 -r 3 A @ x

Память dense: 128000000 байт
Память CSR: 976004 байт
Результаты совпадают: True
CSR:
54.1 us +- 8.86 us per loop (mean +- std. dev. of 3 runs, 20 loops each)
Dense:
1.19 ms +- 110 us per loop (mean +- std. dev. of 3 runs, 20 loops each)


## 8.3

Постройте график (или таблицу) времени умножения A @ x в зависимости от плотности от 0.001 до 1.0 при фиксированном n = 2000. Найдите точку, где sparse перестаёт выигрывать.

densities = [0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0]

In [11]:
import numpy as np
from scipy import sparse

n = 2000
densities = [0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
x = np.ones(n)
times_csr = []
times_dense = []

print("Плотность | CSR, мс | Dense, мс")

for density in densities:
    S = sparse.random(n, n, density=density, format="csr", random_state=1520441)
    A = S.toarray()

    # Прогрев перед измерением.
    S @ x
    A @ x

    csr_time = %timeit -n 20 -r 3 -o -q S @ x
    dense_time = %timeit -n 20 -r 3 -o -q A @ x
    times_csr.append(csr_time.average)
    times_dense.append(dense_time.average)

    print(f"{density:9.3f} | {csr_time.average * 1000:7.3f} | {dense_time.average * 1000:9.3f}")

for i in range(len(densities)):
    if times_csr[i] >= times_dense[i]:
        print("Первая проверенная плотность, где CSR не быстрее:", densities[i])
        if i > 0:
            print("Переход нужно уточнять между", densities[i - 1], "и", densities[i])
        break
else:
    print("На проверенных плотностях CSR быстрее.")

Плотность | CSR, мс | Dense, мс
    0.001 |   0.006 |     0.135
    0.005 |   0.015 |     0.130
    0.010 |   0.021 |     0.118
    0.050 |   0.101 |     0.119
    0.100 |   0.223 |     0.126
    0.200 |   0.466 |     0.127
    0.500 |   1.304 |     0.145
    1.000 |   2.585 |     0.148
Первая проверенная плотность, где CSR не быстрее: 0.1
Переход нужно уточнять между 0.05 и 0.1
